RetailPulse 360

Notebook 11 — Cross-sell / Recommendation Engine

Goal: Build "customers who bought this also bought..." recommendations for Stylo's own
catalog, using real H&M co-purchase behavior (Notebook 03) as the grounding signal — same
methodology as Notebook 01's Rossmann personality transfer: real behavioral PATTERNS carry
over, even though the literal product IDs don't.

Input: cross_sell_pairs.csv, products.csv, skus.csv
Output: cross_sell_recommendations.csv

In [3]:
# 1. IMPORTS
# ============================================================

import pandas as pd
import numpy as np

print("Libraries imported successfully.")

Libraries imported successfully.


In [4]:
# 2. LOAD INPUTS
# ============================================================

BASE_PATH = "/kaggle/input/datasets/hamaz911/notebook-11-datasets/"

cross_sell_pairs = pd.read_csv(BASE_PATH + "cross_sell_pairs.csv")
products = pd.read_csv(BASE_PATH + "products.csv")
skus = pd.read_csv(BASE_PATH + "skus.csv")

print("Loaded:")
for name, df in [("cross_sell_pairs", cross_sell_pairs), ("products", products), ("skus", skus)]:
    print(f"  {name}: {df.shape}")

Loaded:
  cross_sell_pairs: (6422, 15)
  products: (72, 7)
  skus: (3296, 13)


In [5]:
# 3. DATA QUALITY — VALIDATE INPUTS
# ============================================================

print("cross_sell_pairs columns:", list(cross_sell_pairs.columns))

print("\nMissing values:")
for name, df in [("cross_sell_pairs", cross_sell_pairs), ("products", products), ("skus", skus)]:
    m = df.isna().sum()
    if m.sum() > 0:
        print(f"  {name}: {dict(m[m > 0])}")
print("(nothing printed above = no missing values)")

print("\nSimilar-item vs genuine cross-category split (from Notebook 03):")
print(cross_sell_pairs["is_similar_item"].value_counts())

print("\nUnique gender/style combos on the 'a' side:")
print(cross_sell_pairs[["stylo_gender_a", "stylo_style_a"]].drop_duplicates())

print("\ncross_sell_pairs product taxonomy columns confirmed present:",
      all(c in cross_sell_pairs.columns for c in ["stylo_gender_a","stylo_style_a","stylo_gender_b","stylo_style_b"]))

cross_sell_pairs columns: ['article_id_a', 'article_id_b', 'co_purchase_count', 'product_code_a', 'product_code_b', 'same_style_different_variant', 'product_type_name_a', 'colour_group_name_a', 'stylo_gender_a', 'stylo_style_a', 'product_type_name_b', 'colour_group_name_b', 'stylo_gender_b', 'stylo_style_b', 'is_similar_item']

Missing values:
(nothing printed above = no missing values)

Similar-item vs genuine cross-category split (from Notebook 03):
is_similar_item
False    4707
True     1715
Name: count, dtype: int64

Unique gender/style combos on the 'a' side:
     stylo_gender_a stylo_style_a
0           Women's        Casual
7           Women's        Formal
350           Men's        Casual
1081           Kids        Casual
2667           Kids        Formal

cross_sell_pairs product taxonomy columns confirmed present: True


In [7]:
# 4. BUILD CATEGORY-AFFINITY MATRIX (FIXED — canonicalize pair order first)
# ============================================================
# Which article was labeled 'a' vs 'b' was arbitrary (by article_id, not
# category), so "Women's Casual x Women's Formal" was splitting into two
# separate groups by coincidence. Fix: sort each row's two category-tags
# into a consistent order BEFORE grouping, so both directions merge into
# one true total.

genuine_pairs = cross_sell_pairs[cross_sell_pairs["is_similar_item"] == False].copy()

def canonical_pair(row):
    cat_a = (row["stylo_gender_a"], row["stylo_style_a"])
    cat_b = (row["stylo_gender_b"], row["stylo_style_b"])
    return pd.Series(sorted([cat_a, cat_b])[0] + sorted([cat_a, cat_b])[1],
                      index=["gender_1", "style_1", "gender_2", "style_2"])

genuine_pairs[["gender_1", "style_1", "gender_2", "style_2"]] = genuine_pairs.apply(canonical_pair, axis=1)

affinity = genuine_pairs.groupby(
    ["gender_1", "style_1", "gender_2", "style_2"]
)["co_purchase_count"].sum().reset_index()
affinity = affinity.rename(columns={"co_purchase_count": "total_affinity_strength"})

print("Category-affinity matrix (canonicalized, one row per true pair):")
print(affinity.sort_values("total_affinity_strength", ascending=False).to_string(index=False))
print("\nTotal rows (should be far fewer duplicates than before):", len(affinity))

Category-affinity matrix (canonicalized, one row per true pair):
gender_1 style_1 gender_2 style_2  total_affinity_strength
 Women's  Casual  Women's  Casual                    18514
 Women's  Casual  Women's  Formal                    10622
 Women's  Formal  Women's  Formal                     8646
    Kids  Casual     Kids  Casual                      252
   Men's  Casual    Men's  Casual                      223
   Men's  Casual  Women's  Formal                       63
   Men's  Casual  Women's  Casual                       30
    Kids  Formal     Kids  Formal                       21

Total rows (should be far fewer duplicates than before): 8


In [8]:
# 5. GENERATE PRODUCT-LEVEL RECOMMENDATIONS
# ============================================================
# Translate category-level affinity into actual product recommendations
# within Stylo's own catalog.

TOP_N_PER_PRODUCT = 5

recommendations = []
for _, product in products.iterrows():
    p_gender, p_style, p_id = product["gender"], product["category"], product["product_id"]

    # find every affinity row touching this product's category
    matches = affinity[
        ((affinity["gender_1"] == p_gender) & (affinity["style_1"] == p_style)) |
        ((affinity["gender_2"] == p_gender) & (affinity["style_2"] == p_style))
    ].copy()

    if len(matches) == 0:
        continue  # no real affinity signal for this product's category (Sports, Men's Formal)

    # for each matched relationship, figure out the "other side" category
    for _, m in matches.iterrows():
        if m["gender_1"] == p_gender and m["style_1"] == p_style:
            target_gender, target_style = m["gender_2"], m["style_2"]
        else:
            target_gender, target_style = m["gender_1"], m["style_1"]

        candidates = products[
            (products["gender"] == target_gender) & (products["category"] == target_style)
            & (products["product_id"] != p_id)
        ]
        for _, cand in candidates.iterrows():
            recommendations.append({
                "source_product_id": p_id, "source_style_name": product["style_name"],
                "recommended_product_id": cand["product_id"], "recommended_style_name": cand["style_name"],
                "affinity_strength": m["total_affinity_strength"],
            })

rec_df = pd.DataFrame(recommendations)
rec_df = rec_df.sort_values(["source_product_id", "affinity_strength"], ascending=[True, False])
rec_df = rec_df.groupby("source_product_id").head(TOP_N_PER_PRODUCT).reset_index(drop=True)

print("Total recommendation rows:", len(rec_df))
print("Products with at least one recommendation:", rec_df["source_product_id"].nunique(), "of", len(products))
print("\nSample (top recommendations for one product):")
print(rec_df.head(5))

Total recommendation rows: 200
Products with at least one recommendation: 40 of 72

Sample (top recommendations for one product):
  source_product_id     source_style_name recommended_product_id  \
0         PROD-0001  Men's Casual Style 1              PROD-0002   
1         PROD-0001  Men's Casual Style 1              PROD-0003   
2         PROD-0001  Men's Casual Style 1              PROD-0004   
3         PROD-0001  Men's Casual Style 1              PROD-0005   
4         PROD-0001  Men's Casual Style 1              PROD-0006   

  recommended_style_name  affinity_strength  
0   Men's Casual Style 2                223  
1   Men's Casual Style 3                223  
2   Men's Casual Style 4                223  
3   Men's Casual Style 5                223  
4   Men's Casual Style 6                223  


In [9]:
# 6. SAVE OUTPUTS
# ============================================================

rec_df.to_csv("cross_sell_recommendations.csv", index=False)
print("Saved cross_sell_recommendations.csv —", rec_df.shape)

Saved cross_sell_recommendations.csv — (200, 5)


In [10]:
# 7. NOTEBOOK SUMMARY
# ============================================================

print("NOTEBOOK 11 SUMMARY — CROSS-SELL / RECOMMENDATION ENGINE")
print(f"Real category-affinity relationships identified: {len(affinity)}")
print(f"Products with at least one recommendation: {rec_df['source_product_id'].nunique()} of {len(products)}")
print(f"Total recommendation rows: {len(rec_df)}")
print()
print("Honest limitation: Sports (all genders) and Men's/Formal have no real H&M")
print("affinity signal to draw from (same gap documented since Notebook 03) — 32 of 72")
print("products get no cross-sell recommendation at all, rather than a fabricated one.")
print()
print("Also worth noting: Men's Casual's only real relationship is with itself —")
print("its recommendations are same-category ('shop more of this'), not genuine")
print("cross-category upsell, because that's what the real data actually shows.")
print()
print("Output: cross_sell_recommendations.csv")
print("\n✓ Notebook 11 completed successfully.")

NOTEBOOK 11 SUMMARY — CROSS-SELL / RECOMMENDATION ENGINE
Real category-affinity relationships identified: 8
Products with at least one recommendation: 40 of 72
Total recommendation rows: 200

Honest limitation: Sports (all genders) and Men's/Formal have no real H&M
affinity signal to draw from (same gap documented since Notebook 03) — 32 of 72
products get no cross-sell recommendation at all, rather than a fabricated one.

Also worth noting: Men's Casual's only real relationship is with itself —
its recommendations are same-category ('shop more of this'), not genuine
cross-category upsell, because that's what the real data actually shows.

Output: cross_sell_recommendations.csv

✓ Notebook 11 completed successfully.
